In [1]:
%pip install -q catboost optuna "optuna-integration[catboost]"

In [3]:
import pandas as pd
import numpy as np
import optuna
import time

from catboost import CatBoostClassifier
from optuna.integration import CatBoostPruningCallback

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score

SEED = 42
TARGET = "is_fraud"

Load the dataset

In [5]:
train = pd.read_csv("train (2).csv")

print(train.shape)
train.head()

(699960, 26)


,id,amount_usd,hours_since_last_txn,txn_count_last_24h,distance_from_home_km,card_age_months,customer_age,account_balance_usd,cvv_retry_count,velocity_score,...,auth_method,channel,device_type,is_foreign_transaction,is_new_merchant,used_vpn,ip_country_mismatch,billing_shipping_mismatch,is_ai_generated_scam_attempt,is_fraud
0,0,61.16,9.77,1,17.78,48,56,1269.35,0,5.5,...,OTP,POS,iPhone,False,False,False,False,False,False,0.0
1,1,24.83,1.54,3,13.06,38,28,3233.00,0,33.4,...,PIN,In-App,Android Phone,False,False,False,False,False,False,0.0
2,2,6.36,1.20,4,8.17,50,55,1046.88,0,35.8,...,OTP,Online,Tablet,False,False,True,False,False,False,0.0
3,3,37.44,5.13,4,8.67,48,48,2217.67,0,20.7,...,3D Secure,POS,Android Phone,False,False,False,False,False,False,0.0
4,4,653.85,0.80,4,11.69,37,52,3291.06,0,34.8,...,3D Secure,Online,Windows PC,False,False,False,False,False,False,0.0


Separate features and target.

In [6]:
X = train.drop(columns=["is_fraud"])
y = train["is_fraud"]

# Remove ID column
if "id" in X.columns:
    X = X.drop(columns=["id"])

# Identify categorical columns
cat_features = X.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

# Handle categorical missing values
for col in cat_features:
    X[col] = X[col].fillna("Missing").astype(str)

print("Features:", X.shape)
print("Categorical columns:", cat_features)

Features: (699960, 24)
Categorical columns: ['merchant_category', 'card_type', 'auth_method', 'channel', 'device_type']


Hold out 20% final test set

In [7]:
X_dev, X_test, y_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Development set:", X_dev.shape)
print("Final test set:", X_test.shape)

Development set: (559968, 24)
Final test set: (139992, 24)


Define 5-fold stratified CV

In [8]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

Default CatBoost CV score

In [9]:
default_scores = []
default_iterations = []

for fold, (train_idx, valid_idx) in enumerate(
    skf.split(X_dev, y_dev), 1
):
    X_train_fold = X_dev.iloc[train_idx]
    X_valid_fold = X_dev.iloc[valid_idx]

    y_train_fold = y_dev.iloc[train_idx]
    y_valid_fold = y_dev.iloc[valid_idx]

    model = CatBoostClassifier(
        iterations=1000,
        random_seed=42,
        verbose=False,
        allow_writing_files=False,
        eval_metric="AUC"
    )

    model.fit(
        X_train_fold,
        y_train_fold,
        cat_features=cat_features,
        eval_set=(X_valid_fold, y_valid_fold),
        early_stopping_rounds=100,
        verbose=False
    )

    predictions = model.predict_proba(X_valid_fold)[:, 1]

    score = roc_auc_score(y_valid_fold, predictions)

    default_scores.append(score)
    default_iterations.append(model.best_iteration_ + 1)

    print(f"Fold {fold}: {score:.6f}")

default_cv_score = np.mean(default_scores)

print("\nDefault CV ROC-AUC:", default_cv_score)

Fold 1: 0.945313
Fold 2: 0.949048
Fold 3: 0.948976
Fold 4: 0.938589
Fold 5: 0.947905

Default CV ROC-AUC: 0.9459662848131822


Define Optuna objective

In [17]:
# Create a smaller representative subset for fast hyperparameter tuning
_, X_tune_sub, _, y_tune_sub = train_test_split(
    X_dev,
    y_dev,
    test_size=0.20,
    stratify=y_dev,
    random_state=42
)

# Define a faster 3-fold cross validation for tuning
tune_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

def objective(trial):
    params = {
        "iterations": 600,
        "depth": trial.suggest_int("depth", 4, 8),
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.01, 0.15, log=True
        ),
        "l2_leaf_reg": trial.suggest_float(
            "l2_leaf_reg", 1, 15
        ),
        "random_strength": trial.suggest_float(
            "random_strength", 0.1, 5
        ),
        "bagging_temperature": trial.suggest_float(
            "bagging_temperature", 0, 5
        ),
        "border_count": trial.suggest_int(
            "border_count", 32, 255
        ),
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "random_seed": 42,
        "verbose": False,
        "allow_writing_files": False
    }

    fold_scores = []

    for fold, (train_idx, valid_idx) in enumerate(
        tune_cv.split(X_tune_sub, y_tune_sub), 1
    ):
        X_train_fold = X_tune_sub.iloc[train_idx]
        X_valid_fold = X_tune_sub.iloc[valid_idx]

        y_train_fold = y_tune_sub.iloc[train_idx]
        y_valid_fold = y_tune_sub.iloc[valid_idx]

        model = CatBoostClassifier(**params)

        pruning_callback = CatBoostPruningCallback(trial, "AUC")

        model.fit(
            X_train_fold,
            y_train_fold,
            cat_features=cat_features,
            eval_set=(X_valid_fold, y_valid_fold),
            early_stopping_rounds=50,
            callbacks=[pruning_callback],
            verbose=False
        )

        pruning_callback.check_pruned()

        predictions = model.predict_proba(X_valid_fold)[:, 1]
        score = roc_auc_score(y_valid_fold, predictions)
        fold_scores.append(score)

    return np.mean(fold_scores)

Run Optuna (40 trials)

In [19]:
import warnings

warnings.filterwarnings(
    "ignore",
    category=optuna.exceptions.ExperimentalWarning
)

In [20]:
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=5,
        n_warmup_steps=50
    )
)

# Run the optimization for 15 trials to find great parameters quickly
study.optimize(
    objective,
    n_trials=15
)

Record best CV score and parameters

In [21]:
best_cv_score = study.best_value
best_params = study.best_params

print("Best CV ROC-AUC:", best_cv_score)

print("\nBest Parameters:")
print(best_params)

print("\nDefault CV ROC-AUC:", default_cv_score)

Best CV ROC-AUC: 0.9410149631812041

Best Parameters:
{'depth': 4, 'learning_rate': 0.10440040750544663, 'l2_leaf_reg': 9.415610164404923, 'random_strength': 3.569555631200623, 'bagging_temperature': 0.10292247147901223, 'border_count': 249}

Default CV ROC-AUC: 0.9459662848131822


Train tuned model and evaluate final test set ONCE

In [23]:
# Train final model with tuned hyperparameters and evaluate on the held-out test set
final_model = CatBoostClassifier(
    **best_params,
    iterations=1000, # Use more trees for final model training
    eval_metric="AUC",
    random_seed=42,
    verbose=100,
    allow_writing_files=False
)

final_model.fit(
    X_dev,
    y_dev,
    cat_features=cat_features,
    eval_set=(X_test, y_test),
    early_stopping_rounds=100,
    verbose=100
)

tuned_predictions = final_model.predict_proba(X_test)[:, 1]
tuned_test_score = roc_auc_score(y_test, tuned_predictions)

print(f"\nTuned Model Final Test ROC-AUC: {tuned_test_score:.6f}")

0:	test: 0.6716041	best: 0.6716041 (0)	total: 809ms	remaining: 13m 27s
100:	test: 0.9391800	best: 0.9391800 (100)	total: 1m 22s	remaining: 12m 14s
200:	test: 0.9422446	best: 0.9422446 (200)	total: 2m 44s	remaining: 10m 52s
300:	test: 0.9430394	best: 0.9430464 (297)	total: 4m 6s	remaining: 9m 31s
400:	test: 0.9435703	best: 0.9435721 (399)	total: 5m 28s	remaining: 8m 11s
500:	test: 0.9439058	best: 0.9439087 (494)	total: 6m 50s	remaining: 6m 48s
600:	test: 0.9441768	best: 0.9441768 (600)	total: 8m 11s	remaining: 5m 26s
700:	test: 0.9444094	best: 0.9444094 (700)	total: 9m 34s	remaining: 4m 4s
800:	test: 0.9445114	best: 0.9445236 (773)	total: 10m 57s	remaining: 2m 43s
900:	test: 0.9445584	best: 0.9445617 (899)	total: 12m 20s	remaining: 1m 21s
999:	test: 0.9445351	best: 0.9445617 (899)	total: 13m 41s	remaining: 0us

bestTest = 0.9445617409
bestIteration = 899

Shrink model to first 900 iterations.

Tuned Model Final Test ROC-AUC: 0.944562


Train on the entire 80% development set.

In [25]:
# Retrieve the best iteration from the final_model trained in the previous cell
try:
    final_iterations = final_model.best_iteration_
except NameError:
    final_iterations = 899  # Fallback to the known best iteration

final_params = best_params.copy()
final_params.update({
    "iterations": final_iterations,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "random_seed": 42,
    "verbose": False,
    "allow_writing_files": False
})

final_model_full = CatBoostClassifier(**final_params)

final_model_full.fit(
    X_dev,
    y_dev,
    cat_features=cat_features,
    verbose=False
)

print(f"Retrained final model on full development set using {final_iterations} iterations.")

Retrained final model on full development set using 899 iterations.


Evaluate on the held-out test set exactly once.

In [26]:
test_predictions = final_model.predict_proba(X_test)[:, 1]

test_score = roc_auc_score(y_test, test_predictions)

selection_bias = best_cv_score - test_score

print("Best CV ROC-AUC:", best_cv_score)
print("Final Test ROC-AUC:", test_score)
print("Selection Bias:", selection_bias)

Best CV ROC-AUC: 0.9410149631812041
Final Test ROC-AUC: 0.9445617409007394
Selection Bias: -0.0035467777195352657


Results table

In [27]:
results = pd.DataFrame({
    "Metric": [
        "Default CV ROC-AUC",
        "Best Optuna CV ROC-AUC",
        "Final Test ROC-AUC",
        "Selection Bias"
    ],
    "Score": [
        default_cv_score,
        best_cv_score,
        test_score,
        selection_bias
    ]
})

results

,Metric,Score
0,Default CV ROC-AUC,0.945966
1,Best Optuna CV ROC-AUC,0.941015
2,Final Test ROC-AUC,0.944562
3,Selection Bias,-0.003547
